In [1]:
import mlrun

# Loads AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and MLRUN_AWS_ROLE_ARN
from dotenv import load_dotenv
load_dotenv() 

# import os
# https://docs.mlrun.org/en/stable/store/datastore.html#s3
# print(os.environ['AWS_ACCESS_KEY_ID'])
# print(os.environ['AWS_SECRET_ACCESS_KEY'])
# print(os.environ['MLRUN_AWS_ROLE_ARN'])

from pathlib import Path
artifact_path = Path.cwd().parent #/ "mlrun-data/"
artifact_path = str(artifact_path.as_posix()) # convert windows path to unix path
artifact_path = "file://" + artifact_path
#print(artifact_path)
p = mlrun.set_environment(api_path="http://localhost:8080", artifact_path=artifact_path)

project = mlrun.load_project(name='finetune-legal-extractor', context="../") # project yaml must be in this directory
# Verify it loaded correctly by checking its status or printing the config
# print(project.to_yaml())

# Run the workflow

In [2]:
from datetime import datetime
source_path = "s3://legal-llama-data/raw"
version = datetime.now().strftime("%Y%m%d_%H%M")

In [3]:
# Train dataset
runobj = mlrun.run_function("raw-proc",  # use the function name registered in the register_funcs.ipynb file
            inputs={"input_uri": f"{source_path}/train.parquet"},
            params={"label_column": "inference",
                    "artifact_key": f'train_data',
                    "version": version,
                    "output_uri_path": 's3://legal-llama-data/processed_training'},
            local=True
    )

# Validation dataset
runobj = mlrun.run_function("raw-proc", 
            inputs={"input_uri": f'{source_path}/validation.parquet'},
            params={"label_column": "inference",
                    "artifact_key": f'validation_data',
                    "version": version,
                    "output_uri_path": 's3://legal-llama-data/processed_training'},
            local=True
    )

# Test dataset
runobj = mlrun.run_function("raw-proc", 
            inputs={"input_uri": f'{source_path}/test.parquet'},
            params={"label_column": "inference",
                    "artifact_key": f'test_data',
                    "version": version,
                    "output_uri_path": 's3://legal-llama-data/processed_training'},
            local=True
    )


> 2026-05-06 12:24:16,328 [info] Storing function: {"db":"http://localhost:8080","name":"raw-proc-process-raw","uid":"ae882384baa5498dbf1eac88cca2b7c8"}
s3://legal-llama-data/raw/train.parquet processed and written to s3://legal-llama-data/processed_training/20260506_1224


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifact_uris
finetune-legal-extractor,...cca2b7c8,0,May 06 04:24:16,NaT,completed,run,raw-proc-process-raw,kind=localowner=jerrohost=Nitro_53,input_uri,label_column=inferenceartifact_key=train_dataversion=20260506_1224output_uri_path=s3://legal-llama-data/processed_training,,train_data=store://datasets/finetune-legal-extractor/raw-proc-process-raw_train_data#0:20260506_1224@ae882384baa5498dbf1eac88cca2b7c8^5988c20280bf37e9834fb228a3247b535f6e7b16


> 2026-05-06 12:24:30,941 [info] Run execution finished: {"name":"raw-proc-process-raw","status":"completed"}
> 2026-05-06 12:24:30,958 [info] Storing function: {"db":"http://localhost:8080","name":"raw-proc-process-raw","uid":"ab2650b824614614995551bfb6a71844"}
s3://legal-llama-data/raw/validation.parquet processed and written to s3://legal-llama-data/processed_training/20260506_1224


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifact_uris
finetune-legal-extractor,...b6a71844,0,May 06 04:24:31,NaT,completed,run,raw-proc-process-raw,kind=localowner=jerrohost=Nitro_53,input_uri,label_column=inferenceartifact_key=validation_dataversion=20260506_1224output_uri_path=s3://legal-llama-data/processed_training,,validation_data=store://datasets/finetune-legal-extractor/raw-proc-process-raw_validation_data#0:20260506_1224@ab2650b824614614995551bfb6a71844^00bbed66d4edfa1f7fbbb346c0599fb0b25a7df5


> 2026-05-06 12:24:35,457 [info] Run execution finished: {"name":"raw-proc-process-raw","status":"completed"}
> 2026-05-06 12:24:35,472 [info] Storing function: {"db":"http://localhost:8080","name":"raw-proc-process-raw","uid":"635c9c2069674026aa8ca9c9cb3f297c"}
s3://legal-llama-data/raw/test.parquet processed and written to s3://legal-llama-data/processed_training/20260506_1224


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifact_uris
finetune-legal-extractor,...cb3f297c,0,May 06 04:24:35,NaT,completed,run,raw-proc-process-raw,kind=localowner=jerrohost=Nitro_53,input_uri,label_column=inferenceartifact_key=test_dataversion=20260506_1224output_uri_path=s3://legal-llama-data/processed_training,,test_data=store://datasets/finetune-legal-extractor/raw-proc-process-raw_test_data#0:20260506_1224@635c9c2069674026aa8ca9c9cb3f297c^bc3fdef3ac75faa97b8b201e3923fc1af1cd80e2


> 2026-05-06 12:24:40,257 [info] Run execution finished: {"name":"raw-proc-process-raw","status":"completed"}


In [4]:
# For kubeflow pipelines: abandoned because of ghost runs and caching issues
# from datetime import datetime

# run_obj = project.run(
#     name="evaluate_noTrain",
#     arguments={
#         "source_path": "s3://legal-llama-data/raw",
#         "version": datetime.now().strftime("%Y%m%d_%H%M")
#     },
#     local=True,   # Run the pipeline sequence locally
#     watch=True    # Print the progress to the console
# )

## Validation

In [2]:
from IPython.display import display

# Testing direct access to S3 data (underlying s3fs)
# data_uri = "s3://legal-llama-data/raw/test.parquet"
# df = mlrun.get_dataitem(data_uri).as_df() #this reads into a dataframe and only works if the file is a csv/parquet... jsonl does not work
# display(df.head(1))

data_uri = "store://datasets/finetune-legal-extractor/raw-proc-process-raw_test_data:latest"
data_pointer = mlrun.get_dataitem(data_uri)
print(data_pointer.url) # S3 path



s3://legal-llama-data/processed_training/20260506_1224/raw-proc-process-raw/0/test_data.parquet


In [6]:
# Testing data versioning and access to registered datasets in MLRun
data_uri = "store://datasets/finetune-legal-extractor/raw-proc-process-raw_train_data:latest"
#data_uri = "store://datasets/finetune-legal-extractor/raw-proc-process-raw_test_data:20260427_1945"

# Fetch the item and immediately convert it to a Pandas DataFrame
df = mlrun.get_dataitem(data_uri).as_df()
print(df.head(1)['inference'][0])


[{'hypothesis': "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.", 'hypothesis_id': 'nda-11', 'label': 'not_mentioned', 'source_clause': ''}
 {'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.', 'hypothesis_id': 'nda-16', 'label': 'entailment', 'source_clause': '5. All Confidential Information in any form and any medium, including all copies thereof, disclosed to the Recipient shall be returned to UNHCR or destroyed: . (a) if a business relationship is not entered into with UNHCR on or before the date which is three (3) months after the date both Parties have signed the Agreement; or '}
 {'hypothesis': 'Agreement shall not grant Receiving Party any right to Confidential Information.', 'hypothesis_id': 'nda-15', 'label': 'entailment', 'source_clause': '4. Nothing in this Agreement is to be construed as granting the Recipient, by implication or otherwise,

In [7]:
"""
# I blame mlrun for this weird behaviour, instead of the URI on the UI with the ://files path segment, it uses ://datasets
# Now that we are not using 
artifact_uri = "store://datasets/finetune-legal-extractor/raw-proc-process-raw_test_data:latest"

data_item = mlrun.get_dataitem(artifact_uri)
s3_path = data_item.url
print(s3_path) 

import pandas as pd
import io

raw_bytes = data_item.get()
df = pd.read_json(io.BytesIO(raw_bytes), orient="records", lines=True)

inferences = df.head(1)['inference'][0]
for i in inferences:
    print(i)
"""


'\n# I blame mlrun for this weird behaviour, instead of the URI on the UI with the ://files path segment, it uses ://datasets\n# Now that we are not using \nartifact_uri = "store://datasets/finetune-legal-extractor/raw-proc-process-raw_test_data:latest"\n\ndata_item = mlrun.get_dataitem(artifact_uri)\ns3_path = data_item.url\nprint(s3_path) \n\nimport pandas as pd\nimport io\n\nraw_bytes = data_item.get()\ndf = pd.read_json(io.BytesIO(raw_bytes), orient="records", lines=True)\n\ninferences = df.head(1)[\'inference\'][0]\nfor i in inferences:\n    print(i)\n'

In [4]:
artifact = project.get_artifact(key="raw-proc-process-raw_validation_data", tag="20260506_1224") # use the db-key not key
# this wont work project.get_artifact(key="train_data")

print(artifact.get_store_url())
print(artifact.target_path) # points to the source path of the latest version of the artifact
print(artifact.db_key)
print(artifact.key)

store://datasets/finetune-legal-extractor/raw-proc-process-raw_validation_data#0:20260506_1224@ab2650b824614614995551bfb6a71844^00bbed66d4edfa1f7fbbb346c0599fb0b25a7df5
s3://legal-llama-data/processed_training/20260506_1224/raw-proc-process-raw/0/validation_data.parquet
raw-proc-process-raw_validation_data
validation_data


In [9]:
# all datasets
artifacts = project.list_artifacts()
datasets = [artifact for artifact in artifacts if artifact['kind'] == "dataset"]
for i in datasets:
    print(i['metadata']['key'], i['metadata']['tag'])

test_data latest
test_data 20260506_1224
validation_data latest
validation_data 20260506_1224
train_data latest
train_data 20260506_1224


# Deleting artifacts directly

In [10]:
adsasd

NameError: name 'adsasd' is not defined

In [ ]:
# The only way to properly delete a data artifact and its historical versions
import os 

artifact = project.get_artifact(key="raw-proc-process-raw_train_data")
project.delete_artifact(artifact, 
                        deletion_strategy=mlrun.common.schemas.artifact.ArtifactsDeletionStrategies.data_force,
                        secrets={
                            "AWS_ACCESS_KEY_ID": os.environ['AWS_ACCESS_KEY_ID'],
                            "AWS_SECRET_ACCESS_KEY": os.environ['AWS_SECRET_ACCESS_KEY']
                            }
                        )


In [ ]:
# This deletes all artifacts in the database
db = mlrun.get_run_db()
db.del_artifacts(project=project.metadata.name)

print("all artifacts wiped from " + project.metadata.name)

all artifacts wiped from finetune-legal-extractor


In [ ]:
project.spec.get_code_path()

'../'